# GPU Portfolio Bench — Demo Notebook

End-to-end walkthrough:
1. Fetch price data
2. CPU vs GPU Monte Carlo VaR — side-by-side timing
3. Efficient frontier (Markowitz)
4. LSTM forecasting model — quick train + inference
5. Custom CUDA kernel comparison

Run on Google Colab with a T4/A100 GPU for best results.

In [ ]:
# Clone & install (uncomment when running on Colab)
# !git clone https://github.com/scotthnguyen/gpu-portfolio-bench.git
# %cd gpu-portfolio-bench
# !pip install -q -r requirements.txt
# !pip install -q cupy-cuda12x  # match your Colab CUDA version

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Data pipeline

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
from src.data.fetch import fetch_prices, compute_log_returns

prices = fetch_prices()
returns = compute_log_returns(prices).dropna(axis=1)
print(f"{returns.shape[1]} assets × {returns.shape[0]} trading days")
returns.tail(3)

## 2. CPU vs GPU Monte Carlo VaR

In [ ]:
from src.models.var_cpu import compute_var_cvar_cpu
from src.models.var_gpu import compute_var_cvar_gpu

N_ASSETS = 50
N_PATHS = 1_000_000

assets = list(returns.columns[:N_ASSETS])
R = returns[assets].to_numpy(dtype=np.float64)
weights = np.ones(N_ASSETS) / N_ASSETS

cpu_r = compute_var_cvar_cpu(R, weights, n_paths=N_PATHS)
print(f"CPU  VaR={cpu_r['VaR']:.4f}  elapsed={cpu_r['elapsed_sec']:.2f}s  "
      f"throughput={cpu_r['throughput_paths_per_sec']:,.0f} paths/s")

if torch.cuda.is_available():
    gpu_r = compute_var_cvar_gpu(R, weights, n_paths=N_PATHS, device='cuda')
    speedup = cpu_r['elapsed_sec'] / gpu_r['elapsed_sec']
    print(f"GPU  VaR={gpu_r['VaR']:.4f}  elapsed={gpu_r['elapsed_sec']:.2f}s  "
          f"throughput={gpu_r['throughput_paths_per_sec']:,.0f} paths/s")
    print(f"\n→ GPU speedup: {speedup:.1f}×  |  GPU memory: {gpu_r['gpu_mem_mb']:.1f} MB")

## 3. Efficient frontier

In [ ]:
import plotly.graph_objects as go
from src.models.portfolio_opt import build_efficient_frontier

N_OPT = 20  # keep small for notebook speed
ef = build_efficient_frontier(
    returns[assets[:N_OPT]].to_numpy(dtype=np.float64),
    assets[:N_OPT],
    n_points=30,
    use_gpu=torch.cuda.is_available(),
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ef.volatilities, y=ef.expected_returns,
    mode='lines+markers',
    marker=dict(color=ef.sharpe_ratios, colorscale='Viridis', showscale=True,
                colorbar=dict(title='Sharpe')),
    text=[f'Sharpe: {s:.2f}' for s in ef.sharpe_ratios],
    name='Efficient frontier',
))
fig.update_layout(title='Efficient Frontier', xaxis_title='Annualized Volatility',
                  yaxis_title='Annualized Return')
fig.show()

## 4. LSTM return forecaster

In [ ]:
import plotly.express as px
import pandas as pd
from src.models.forecaster import train, load_model, predict_next_day
from pathlib import Path

N_FORECAST_ASSETS = 10
forecast_assets = assets[:N_FORECAST_ASSETS]
R_fore = returns[forecast_assets].to_numpy(dtype='float32')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt_path = Path('../results/checkpoints/lstm_best.pt')

if ckpt_path.exists():
    print("Loading existing checkpoint...")
    model, ckpt = load_model(str(ckpt_path), device=device)
    print(f"Loaded epoch {ckpt['epoch']}, val loss {ckpt['val_loss']:.6f}, assets: {ckpt['asset_names']}")
    # Use its assets if they match
    if ckpt['asset_names'] == forecast_assets:
        pass  # already loaded
    else:
        print("Asset mismatch — retraining on current selection")
        result = train(R_fore, forecast_assets, epochs=30, hidden=64, device=device)
        model, ckpt = load_model(result.checkpoint_path, device=device)
else:
    print(f"Training LSTM on {N_FORECAST_ASSETS} assets, device={device}")
    result = train(R_fore, forecast_assets, epochs=30, hidden=64, device=device)
    model, ckpt = load_model(result.checkpoint_path, device=device)

# Training curves (if we just trained)
loss_path = Path('../results/train_losses.csv')
if loss_path.exists():
    ldf = pd.read_csv(loss_path)
    px.line(ldf, x='epoch', y=['train_loss', 'val_loss'], title='LSTM training curves').show()

pred = predict_next_day(model, R_fore[-ckpt['window']:], device=device)
pred_df = pd.DataFrame({'Asset': forecast_assets, 'Predicted daily return': pred,
                        'Annualized (%)': pred * 252 * 100})
pred_df.sort_values('Predicted daily return', ascending=False)

## 5. LSTM → Markowitz: full ML pipeline

Use LSTM-predicted next-day returns as `μ` instead of historical means, then solve for max-Sharpe weights. This is the end-to-end GPU ML lifecycle: **data → train → infer → optimize**.

In [ ]:
from src.models.forecast_optimizer import forecast_weights
from src.models.portfolio_opt import max_sharpe_weights
import plotly.graph_objects as go

N_PIPE = 20  # assets for this pipeline demo
pipe_assets = list(returns.columns[:N_PIPE])
R_pipe = returns[pipe_assets].to_numpy(dtype=np.float64)

# Historical max-Sharpe (baseline)
mu_hist = R_pipe.mean(axis=0) * 252
cov_ann = np.cov(R_pipe.T) * 252
w_hist = max_sharpe_weights(mu_hist, cov_ann)

# LSTM-enhanced max-Sharpe
result_lstm = forecast_weights(
    R_pipe.astype(np.float32),
    pipe_assets,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
w_lstm = result_lstm['weights']

# Compare allocation
fig = go.Figure()
fig.add_bar(x=pipe_assets, y=w_hist * 100, name='Historical μ (Markowitz)')
fig.add_bar(x=pipe_assets, y=w_lstm * 100, name='LSTM-predicted μ')
fig.update_layout(
    title='Portfolio Weights: Historical vs LSTM-Enhanced Optimization',
    xaxis_title='Asset', yaxis_title='Weight (%)', barmode='group',
)
fig.show()

# Predicted returns comparison
fig2 = go.Figure()
fig2.add_bar(x=pipe_assets, y=mu_hist * 100, name='Historical annualized μ (%)')
fig2.add_bar(x=pipe_assets, y=result_lstm['predicted_annual_returns'] * 100, name='LSTM predicted μ (%)')
fig2.update_layout(title='Return Forecasts: Historical Mean vs LSTM', barmode='group',
                   yaxis_title='Annualized return (%)')
fig2.show()

print(f"Top 5 LSTM weights: {sorted(zip(pipe_assets, w_lstm), key=lambda x: -x[1])[:5]}")

## 6. Custom CUDA kernel (CuPy raw kernel)

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

sweep_path = Path('../results/benchmark_sweep.csv')
if sweep_path.exists():
    sweep = pd.read_csv(sweep_path)
    sweep['n_assets'] = sweep['n_assets'].astype(str) + ' assets'

    fig = px.line(
        sweep[sweep['device'] == 'cpu'],
        x='n_paths', y='throughput_paths_per_sec',
        color='n_assets', log_x=True, log_y=True, markers=True,
        labels={'n_paths': 'Monte Carlo paths', 'throughput_paths_per_sec': 'Paths / second'},
        title='CPU Throughput: paths/sec vs simulation scale',
    )
    fig.show()

    if 'cuda' in sweep['device'].values:
        cpu_df = sweep[sweep['device']=='cpu'][['n_assets','n_paths','elapsed_sec']].rename(columns={'elapsed_sec':'cpu_sec'})
        gpu_df = sweep[sweep['device']=='cuda'][['n_assets','n_paths','elapsed_sec']].rename(columns={'elapsed_sec':'gpu_sec'})
        speedup_df = pd.merge(cpu_df, gpu_df, on=['n_assets','n_paths'])
        speedup_df['speedup'] = speedup_df['cpu_sec'] / speedup_df['gpu_sec']
        fig2 = px.line(
            speedup_df, x='n_paths', y='speedup', color='n_assets',
            log_x=True, markers=True,
            title='GPU Speedup vs CPU (run on GPU instance for this chart)',
        )
        fig2.add_hline(y=1, line_dash='dash', line_color='gray', annotation_text='break-even')
        fig2.show()
    else:
        print("Run on a GPU instance to see speedup chart. GPU data not in results yet.")
else:
    print("Run: python -m src.bench.run_sweep --device cpu  first")

In [ ]:
try:
    from src.models.var_cuda_kernel import compute_var_cvar_cupy
    cupy_r = compute_var_cvar_cupy(R, weights, n_paths=N_PATHS)
    speedup_vs_cpu = cpu_r['elapsed_sec'] / cupy_r['elapsed_sec']
    print(f"CuPy kernel  VaR={cupy_r['VaR']:.4f}  elapsed={cupy_r['elapsed_sec']:.2f}s")
    print(f"Speedup vs CPU: {speedup_vs_cpu:.1f}×")
    print(f"GPU memory: {cupy_r['gpu_mem_mb']:.1f} MB")
except RuntimeError as e:
    print(f"CuPy not available on this machine: {e}")
    print("Install with: pip install cupy-cuda12x  (match your CUDA version)")

## 7. Run the full benchmark sweep

This writes `results/benchmark_sweep.csv` — feed it into the Streamlit dashboard.

```bash
# CPU only (works anywhere)
python -m src.bench.run_sweep --device cpu

# GPU + CPU (on a cloud instance)
python -m src.bench.run_sweep --device all

# Launch dashboard
streamlit run src/dashboard/app.py
```

In [ ]:
try:
    from src.models.var_cuda_kernel import compute_var_cvar_cupy
    cupy_r = compute_var_cvar_cupy(R, weights, n_paths=N_PATHS)
    speedup_vs_cpu = cpu_r['elapsed_sec'] / cupy_r['elapsed_sec']
    print(f"CuPy kernel  VaR={cupy_r['VaR']:.4f}  elapsed={cupy_r['elapsed_sec']:.2f}s")
    print(f"Speedup vs CPU: {speedup_vs_cpu:.1f}×")
    print(f"GPU memory: {cupy_r['gpu_mem_mb']:.1f} MB")
except RuntimeError as e:
    print(f"CuPy not available: {e}")

## 6. Run the full benchmark sweep

This writes `results/benchmark_sweep.csv` — feed it into the Streamlit dashboard.

```bash
python -m src.bench.run_sweep --device all
streamlit run src/dashboard/app.py
```